# GFN2 fine-tune of MACE-OFF on 250 rotaxane frames (Colab)

Runs the same workflow as the 10-frame macOS probe, at 250 frames and on a GPU:

1. **GFN2-xTB labels** for `data/rot1_sampled_250.xyz` (torch-free subprocess)
2. **Reference pool** for the latent-OOD detector, built from the MACE-OFF23 test split with *stock* `off-medium`
3. **Baseline (before)**: per-frame molecule-mean OOD for all 250 frames
4. **Fine-tune** `off-medium` on the GFN2 labels (energy **and** forces — the probe showed an energy-only hammer shifts the whole latent manifold)
5. **After**: rebuild the pool *through the finetuned encoder* (latent spaces are not comparable across checkpoints) and re-score, plus a **size-matched stock-pool control** (7b) that makes the comparison like-for-like
6. **Per-atom OOD maps** before/after for one frame
7. **Save the checkpoint + scores to Google Drive**

**First: Runtime → Change runtime type → GPU.** Total wall time ≈ 60–90 min on a
T4 (an A100 does the fine-tune itself in minutes). If the pip cell changes
`torch`/`numpy` versions, Runtime → Restart session, then re-run it (the clone
is skipped on re-run).

In [ ]:
# @title 1. Clone the repo + install the stack {display-mode:"form"}
import os, subprocess, torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU attached — the fine-tune cell will be ~10x slower.")

if not os.path.exists("/content/mace/code/mace_calc.py"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/MauricioCafiero/MACE_UseAndTrain.git",
                    "/content/mace"], check=True)
    print("repo cloned")
%cd /content/mace
!pip -q install mace-torch tblite 2>&1 | tail -3
print("install done")

### What the probe taught us (carried into the defaults below)

On the 10-frame probe (energy-only fine-tune, 50 epochs), the unusual-chemistry score went **UP** ~3×: mean OOD 0.150 → 0.455 vs the original pool, and 0.256 even vs a pool rebuilt through the finetuned encoder. Decomposition: ~+0.20 wholesale latent-space shift + ~+0.10 genuine novelty — under an energy-dominated hammer the model moves off the OFF23 manifold entirely.

So this run changes three things: **forces carry real weight** (`forces_weight=100`, not 1), **fewer epochs** (30, with EMA + plateau LR decay), and the **self-pool comparison** (finetuned model vs pool built through the finetuned encoder) is the headline number.

In [ ]:
# @title 2. GFN2-xTB labels for the 250 frames (~3 min) {display-mode:"form"}
# tblite and torch bundle separate OpenMP runtimes and segfault when loaded
# into one process, so labeling runs as its own torch-free worker pool
# (code/gfn2_label.py never imports torch).
!mkdir -p runs
!python code/gfn2_label.py data/rot1_sampled_250.xyz runs/rot250_gfn2.xyz --workers 4

In [ ]:
# @title 3. Split into train / valid (every 10th frame held out) {display-mode:"form"}
import sys; sys.path.insert(0, "code")
from ase.io import read, write

frames = read("runs/rot250_gfn2.xyz", index=":")
valid = [a for i, a in enumerate(frames) if i % 10 == 0]
train = [a for i, a in enumerate(frames) if i % 10 != 0]
write("runs/rot250_gfn2_train.xyz", train, format="extxyz")
write("runs/rot250_gfn2_valid.xyz", valid, format="extxyz")
print(f"{len(train)} train / {len(valid)} valid frames")

In [ ]:
# @title 4. Build the stock off-medium reference pool (one-time, ~15 min) {display-mode:"form"}
# Downloads the MACE-OFF23 test split (~81 MB) and encodes 2500 drug-like
# frames into the latent pool that trust.py / ood_map.py score against.
import sys; sys.path.insert(0, "code")
from activation_ood import ReferencePool

pool = ReferencePool.build(n_frames=2500, out="data/off23_pool.npz")
print("pool atoms:", pool.atom_vecs.shape[0])

In [ ]:
# @title 5. Baseline: score all 250 frames with stock off-medium ("before", ~2 min) {display-mode:"form"}
import numpy as np
import mace_calc as mc
from trust_frames import read_frames
from activation_ood import atom_ood_scores

frames = read_frames("data/rot1_sampled_250.xyz")
mc.attach(frames[0], model="off-medium", device="cuda", dtype="float32")
shared = frames[0].calc            # one calculator shared across all frames

before = []
for k, at in enumerate(frames):
    at.calc = shared
    d = atom_ood_scores(at, pool)["distances"]
    before.append(float(np.nanmean(d)))
    if k % 50 == 0:
        print(f"frame {k:3d}  mean OOD {before[-1]:.3f}")
before = np.array(before)
print(f"\nSTOCK off-medium: mean {before.mean():.3f}  "
      f"range [{before.min():.3f}, {before.max():.3f}]")

In [ ]:
# @title 6. Fine-tune off-medium on the GFN2 labels (GPU, ~30–60 min) {display-mode:"form"}
from finetune_mace import run_finetune, find_latest_model

run_finetune(
    "runs/rot250_gfn2_train.xyz", "runs/rot250_gfn2_valid.xyz",
    foundation_model="off-medium", name="rot250", results_dir="runs",
    max_num_epochs=30,
    energy_weight=100.0, forces_weight=100.0,
    scheduler="ReduceLROnPlateau",
    device="cuda", default_dtype="float32",
    extra=("--batch_size=8", "--valid_batch_size=8", "--eval_interval=2"),
)
model_path = find_latest_model("runs", "rot250")
print("finetuned checkpoint:", model_path)

In [ ]:
# @title 7. Rebuild the pool through the finetuned encoder + re-score ("after") {display-mode:"form"}
# Latent spaces are not comparable across checkpoints, so the finetuned model
# is scored against a pool built through the SAME encoder (the stage-5 control
# from the probe). mc.get_calculator is monkeypatched to hand out the
# finetuned calculator everywhere a pool build asks for one.
#
# NOTE on the "frame k failed (16 is not in list); skipping" warnings: the
# finetuned checkpoint's atomic-energies table only contains the elements it
# was TRAINED on (H,C,N,O,F), so OFF23 reference frames containing P/S/Cl/Br/I
# cannot be encoded and are skipped. That is expected -- the surviving pool is
# CHNOF-only and smaller than the stock pool. Cell 7b builds a size-matched
# stock-model control so the before/after comparison stays like-for-like.
from pathlib import Path
from mace.calculators import MACECalculator
import mace_calc as mc
import matplotlib.pyplot as plt

ft = MACECalculator(model_paths=str(model_path), device="cuda",
                    default_dtype="float32")
stock_get_calculator = mc.get_calculator   # keep for the 7b control
mc.get_calculator = lambda **kw: ft   # pool build runs the finetuned encoder

ft_pool = ReferencePool.build(n_frames=2500, out="data/off23_pool_ft.npz")

after = []
for k, at in enumerate(frames):
    at.calc = ft
    after.append(float(np.nanmean(atom_ood_scores(at, ft_pool)["distances"])))
after = np.array(after)

plt.figure(figsize=(7.5, 4))
plt.hist(before, bins=30, alpha=0.55,
         label=f"stock off-medium / own pool  (mean {before.mean():.3f})")
plt.hist(after, bins=30, alpha=0.55,
         label=f"finetuned / own pool  (mean {after.mean():.3f})")
plt.axvline(0.25, ls="--", c="k", lw=1)
plt.text(0.255, plt.ylim()[1] * 0.9, "trust line (0.25)", fontsize=8)
plt.xlabel("molecule-mean latent OOD (cosine)")
plt.ylabel("frames")
plt.title("250 rotaxane frames: before vs after GFN2 fine-tune")
plt.legend(fontsize=8)
plt.show()

print(f"{'':<12}{'before':>8}{'after':>8}")
print(f"{'mean':<12}{before.mean():8.3f}{after.mean():8.3f}")
print(f"{'max':<12}{before.max():8.3f}{after.max():8.3f}")
print(f"{'frames>0.25':<12}{int((before > 0.25).sum()):8d}"
      f"{int((after > 0.25).sum()):8d}")

In [ ]:
# @title 7b. Size-matched control pool — the like-for-like "before" {display-mode:"form"}
# Cell 7's ft pool is CHNOF-only and much smaller than the stock pool (the
# P/S/Cl/Br/I frames were skipped). Nearest-neighbour distances INFLATE as
# pools shrink, so some of the before/after rise is a pool artifact, not
# learning. Control: rebuild a STOCK-model pool with the same element filter
# and frame count, re-score "before" against it, and compare that to "after".
import activation_ood as ao

mc.get_calculator = stock_get_calculator          # stock encoder again
ALLOWED = frozenset({1, 6, 7, 8, 9})              # H, C, N, O, F
_orig_iter = ao._iter_frames
def _chnof_iter(xyz_dir):
    for fr in _orig_iter(xyz_dir):
        if set(fr.get_atomic_numbers()) <= ALLOWED:
            yield fr
ao._iter_frames = _chnof_iter

stock_pool_matched = ReferencePool.build(n_frames=2500,
                                         out="data/off23_pool_CHNOF.npz")
ao._iter_frames = _orig_iter                      # undo the filter

before_matched = []
for k, at in enumerate(frames):
    at.calc = shared
    before_matched.append(
        float(np.nanmean(atom_ood_scores(at, stock_pool_matched)["distances"])))
before_matched = np.array(before_matched)

print(f"{'':<24}{'pool atoms':>12}{'mean OOD':>10}")
print(f"{'stock / big pool':<24}{pool.atom_vecs.shape[0]:12d}{before.mean():10.3f}")
print(f"{'stock / CHNOF-matched':<24}{stock_pool_matched.atom_vecs.shape[0]:12d}"
      f"{before_matched.mean():10.3f}")
print(f"{'finetuned / ft pool':<24}{ft_pool.atom_vecs.shape[0]:12d}{after.mean():10.3f}")
print(f"\nlike-for-like elevation (ft vs stock encoder, size-matched "
      f"CHNOF pools): {after.mean() - before_matched.mean():+.3f}")

In [ ]:
# @title 8. Per-atom OOD maps, before vs after (frame 0) {display-mode:"form"}
# Before: the stock ood_map.py CLI (scores with off-medium + stock pool).
!python code/ood_map.py data/rot1_sampled_250.xyz --model off-medium --frame 0 --out-dir viz_before

# After: same tool in-process, pointed at the finetuned checkpoint + its pool.
from pathlib import Path
import trust
trust.POOLS["rot250-ft"] = Path("data/off23_pool_ft.npz")  # ood_map shares this dict
import ood_map
mc.get_calculator = lambda **kw: ft          # ood_map's mc is this same module
ood_map.main(["data/rot1_sampled_250.xyz", "--model", "rot250-ft",
              "--frame", "0", "--out-dir", "viz_after"])

from IPython.display import Image, display
print("\nBEFORE (stock):"), display(Image("viz_before/frame00_ood.png"))
print("AFTER (finetuned):"), display(Image("viz_after/frame00_ood.png"))

In [ ]:
# @title 9. Save the checkpoint + scores to Google Drive {display-mode:"form"}
# (Replace cell 9's files.download: artifacts survive in Drive even if the
# Colab VM dies, and can be picked up from any computer.)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil

dst = "/content/drive/MyDrive/mace_rot250"
os.makedirs(dst, exist_ok=True)
shutil.copy(model_path, f"{dst}/{model_path.name}")
results = {"before": before, "after": after}
if "before_matched" in globals():          # cell 7b ran
    results["before_matched"] = before_matched
np.savez(f"{dst}/rot250_ood_scores.npz", **results)
print("saved to", dst, ":", sorted(os.listdir(dst)))

### Reading the result

- **The like-for-like comparison is cell 7b's `after` vs `before_matched`** — both scored against CHNOF-only pools of the same size. The headline `after` vs big-pool `before` comparison mixes in a pool-size artifact: nearest-neighbour distances inflate as pools shrink (on the 10-frame probe this was most of the apparent effect — stock vs size-matched CHNOF pool scored 0.222, not 0.149, and the finetuned 0.256 vs that is only +0.03, not +0.10).
- **Why step 7 skips frames** (`frame k failed (16 is not in list)`): the finetuned checkpoint's atomic-energies table contains only its training elements (H,C,N,O,F), so OFF23 reference frames with P/S/Cl/Br/I can't be encoded. Expected; cell 7b compensates. If you want a finetuned model that keeps all 10 OFF23 elements, pass an explicit `--E0s` dict covering all of them (see `finetune_mace.compute_e0s_gfn2`) instead of `--E0s=average`.
- **Unusual ≠ unreliable.** The per-atom/localization score flags chemistry that is far from the OFF23 training distribution; the molecule-mean verdict (≤ 0.25 → TRUST) is the reliability pre-filter. Judge the fine-tune by the mean, look at the maps for *where*.
- **Scoring dtype.** Everything on Colab runs float32 for speed (the macOS probe used float64). Cosine distances shift at the ~0.001 level — far below the 0.25 threshold — but don't mix f32 scores with the f64 numbers in OOD_NOTES.md.
- **Knobs worth touching for a second run:** `max_num_epochs` (30 → 15–20 if the after-mean drifts up and validation loss plateaus early), `forces_weight` (100 → 1000 for an even stiffer PES), and the `i % 10` split ratio. On an A100 the whole fine-tune takes minutes, so bigger sweeps are cheap.
- **Parity check (optional):** absolute energies of stock MACE, the finetuned model, and GFN2 sit on different E0 baselines and must never be compared directly — only differences (ΔE between frames within one model) or held-out validation RMSE from the training log are meaningful.